# To do:

- [ ] Iterate over all molecules instead of just the first 3 in filenames (i.e remove the [:3] after filename in the for loop...)
- [ ] Make cool plots that look like the original DDCC plots
- [ ] Make this a python file and run in Narval with ALL models for all basis sets  ONLY IF NOT ENOUGH RAM TO RUN LOCALLY

# 

In [1]:
import psutil
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from sklearn.metrics import mean_absolute_error, r2_score

from glob import glob
import psi4
from helper_CC_ML_spacial import HelperCCEnergy
# from helper_CC_ML import *

from glob import glob
import random
random.seed(0)


  Threads set to 12 by Python driver.


In [2]:
# properties=['Evir1', 'Hvir1', 'Jvir1', 'Kvir1', 'Evir2', 'Hvir2', 'Jvir2', 'Kvir2', 'Eocc1', 'Jocc1', 'Kocc1', 'Hocc1','Eocc2', 'Jocc2', 'Kocc2', 'Hocc2', 'Jia1', 'Jia2', 'Kia1', 'Kia2','diag', 'orbdiff', 'doublecheck', 't2start', 't2mag', 't2sign', 'Jia1mag', 'Jia2mag','Kia1mag', 'Kia2mag','t2']
properties=['Evir1', 'Hvir1', 'Jvir1', 'Kvir1', 'Evir2', 'Hvir2', 'Jvir2', 'Kvir2', 'Eocc1', 'Jocc1', 'Kocc1', 'Hocc1','Eocc2', 'Jocc2', 'Kocc2', 'Hocc2', 'Jia1', 'Jia2', 'Kia1', 'Kia2','diag', 'orbdiff', 'doublecheck', 't2start', 't2mag', 't2sign', 'Jia1mag', 'Jia2mag','Kia1mag', 'Kia2mag']

In [3]:
with open('out_of_data.txt','r') as f:
    lines = f.readlines()
    
filenames = [element[:-1] for element in lines]
filenames[:4]

['../machine_learning/data/methane72.xyz',
 '../machine_learning/data/ammonia159.xyz',
 '../machine_learning/data/methane5.xyz',
 '../machine_learning/data/water91.xyz']

In [10]:
path_to_change = '/mnt/c/Users/Maxim/Documents/Toronto/Research/code/out/'

# ML-predicted initialisation

## all zeros for t1-amplitudes, use ML-predicted t2-amplitiudes

In [ ]:
basis_sets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']
best5=["doublecheck","t2start","t2mag","orbdiff","diag"]

n_list = [20, 40, 60, 80, 100] # iterating over N=20,40,60,80,100 training molecules

for basis in basis_sets:
    for n in n_list:
        print(f"Processing N={n} molecules with {basis} basis set")

        model_path = path_to_change + f'faster_{basis}_model_{n}.pkl'
        data_split_path = path_to_change + f'faster_{basis}_data_splits_{n}.pkl'
        print(model_path)
        print(data_split_path)

        with open(model_path,'rb') as m:
            model = joblib.load(m)            # import model
        
        ml_iter_list = []
        ml_dev = []
        ml_time_to_converge_list = []
        for fn in filenames[:3]:
            print(f"Processing {fn}")       # import xyz file
            with open(fn,'r') as f:
                text=f.read()
            mol = psi4.geometry(text)                
            
            psi4.core.clean()
            psi4.core.be_quiet()
            
            psi4.set_options({'basis': basis,
                              'scf_type':     'pk',
                              'reference':    'rohf',
                              'mp2_type':     'conv',
                              'e_convergence': 1e-8,
                              'd_convergence': 1e-8})

            rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)        # rhf_e is Hartree-Fock energy 
            scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
            
            A=HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=True)

            # Here we are initializing with the ML predicted t2-amplitudes, first we predict using the top 5 features
            y_pred = model.predict(np.vstack([getattr(A,i).flatten() for i in best5]).T)
        
            A.t1 = np.zeros((A.t1.shape))
            A.t2 = y_pred.reshape(*A.t2.shape)
            A.t2start = y_pred.reshape(*A.t2.shape)

            mlE_0 = A.compute_energy(iterate=False)   # ML-predicted energy
            mlE = A.compute_energy()                # exact CCSD energy
                    
            ml_dev.append(abs(mlE_0-mlE))  # energy deviation from CCSD
            
            mlHistory = A.history               # tuple, (iteration_nunber, CCSDcorr_E)
            mlHistory.insert(0, (0,mlE_0))     # inserting the INITIAL t2 amplitude from ML method
            
            #for i in mlHistory:
            #    print(i)
            #print("\n")
            
            iterations = len(mlHistory)     # number of iterations to converge
            #print(f"No. of iterations to converge: {iterations}")
            ml_iter_list.append(iterations)
            
            ml_time_to_converge = A.time_to_converge 
            ml_time_to_converge_list.append(ml_time_to_converge)
            #print(f"Time to converge: {ml_time_to_converge}")
            
            features = pd.DataFrame(np.array([getattr(A,attr).flatten() for attr in properties]).T,columns=properties)
            #print(f"Features: \n{features}")

        print("\n")
        print("Iterations for each molecule list: ", ml_iter_list)
        print("Mean iterations to converge: ", np.mean(ml_iter_list))
        print("Deviations from CCSD for each molecule: ", ml_dev)
        print(f"Mean deviation from CCSD: {np.mean(ml_dev)} (Eₕ)")
        print("Times to converge for each molecule: ", ml_time_to_converge_list)
        print("Mean time to converge", np.mean(ml_time_to_converge_list), " seconds")
        print("Next ...\n")


Processing N=20 molecules with STO-3G basis set
/mnt/c/Users/Maxim/Documents/Toronto/Research/code/out/faster_STO-3G_model_20.pkl
/mnt/c/Users/Maxim/Documents/Toronto/Research/code/out/faster_STO-3G_data_splits_20.pkl


/home/max/miniconda3/envs/quantum/lib/python3.10/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/max/miniconda3/envs/quantum/lib/python3.10/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.6.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/max/miniconda3/envs/quantum/lib/python3.10/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor 

Processing ../machine_learning/data/methane72.xyz
Computing RHF reference.


/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 9 basis functions.
(9, 9)
(8, 8)
Building initial guess...

..initialized CCSD in 0.033 seconds.

CCSD Iteration   0: CCSD correlation = -0.076955765884490   dE =  7.69558E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.076955765884490   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.002 seconds!
CCSD Iteration   0: CCSD correlation = -0.076955765884490   dE =  7.69558E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.077423087857428   dE = -4.67322E-04   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.078269001172038   dE = -8.45913E-04   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.078839487238039   dE = -5.70486E-04   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.079042760982575   dE = -2.03274E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.079051616408895   dE = -8.85543E-06   DIIS = 4
CCSD Iteration   6: CCSD correlation

...

Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 8 basis functions.
(8, 8)
(7, 7)
Building initial guess...

..initialized CCSD in 0.200 seconds.

CCSD Iteration   0: CCSD correlation = -0.065028510098437   dE =  6.50285E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.065028510098437   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.002 seconds!
CCSD Iteration   0: CCSD correlation = -0.065028510098437   dE =  6.50285E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.064659669075085   dE =  3.68841E-04   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.064780267704599   dE = -1.20599E-04   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.064889134528916   dE = -1.08867E-04   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.064942615696784   dE = -5.34812E-05   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.064944837673442   dE = -2.22198E-06   DIIS = 4
CCSD Iteration   6: CCSD correlation

/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 9 basis functions.
(9, 9)
(8, 8)
Building initial guess...

..initialized CCSD in 0.034 seconds.

CCSD Iteration   0: CCSD correlation = -0.078280143745861   dE =  7.82801E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.078280143745861   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.002 seconds!
CCSD Iteration   0: CCSD correlation = -0.078280143745861   dE =  7.82801E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.077261844650082   dE =  1.01830E-03   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.078012011818014   dE = -7.50167E-04   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.078581790888784   dE = -5.69779E-04   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.078851392032094   dE = -2.69601E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.078865188991746   dE = -1.37970E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation

/home/max/miniconda3/envs/quantum/lib/python3.10/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/max/miniconda3/envs/quantum/lib/python3.10/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.6.1 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/max/miniconda3/envs/quantum/lib/python3.10/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor 

Processing ../machine_learning/data/methane72.xyz


# Normal initialisation

## all zeros for t1-amplitudes, and using the MP2 t2 amplitudes

In [7]:
basis_sets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

for basis in basis_sets:
    print(f"Basis: {basis}")
    # Normal initialisation i.e. all zeros for t1-amplitudes, and using the MP2 t2 amplitudes
    normal_iter_list = []
    normal_dev = []
    normal_time_to_converge_list = []
    
    for fn in filenames[:3]:
        print(f"Processing {fn}")      # import xyz file
        with open(fn,'r') as f:
            text=f.read()
        mol = psi4.geometry(text)                
        
        psi4.core.clean()
        psi4.core.be_quiet()
        
        psi4.set_options({'basis': basis,
                          'scf_type':     'pk',
                          'reference':    'rohf',
                          'mp2_type':     'conv',
                          'e_convergence': 1e-8,
                          'd_convergence': 1e-8})

        rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)        # rhf_e is Hartree-Fock energy 
        scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
        
        A=HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=True)

        # Here is the "normal" way to initialize i.e. NOT Machine Learning predicted, NOT random, NOT all zeros
        MP2T2=A.t2start
        A.t1 = np.zeros((A.t1.shape))
        A.t2 = MP2T2
        
        MP2E = A.compute_energy(iterate=False)                    # MP2 (initial) energy
        CCSDE = A.compute_energy()                                   # exact CCSD energy
    
        normal_dev.append(abs(MP2E-CCSDE))  # energy deviation from CCSD
        
        normalHistory = A.history               # tuple, (iteration_nunber, CCSDcorr_E)
        normalHistory.insert(0, (0,MP2E))     # inserting the INITIAL t2 amplitude from HF method
        
        #for i in normalHistory:
        #    print(i)
        #print("\n")
        
        iterations = len(normalHistory)     # number of iterations to converge
        #print(f"No. of iterations to converge: {iterations}")
        normal_iter_list.append(iterations)
        
        normal_time_to_converge = A.time_to_converge 
        normal_time_to_converge_list.append(normal_time_to_converge)
        #print(f"Time to converge: {normal_time_to_converge}")
        
        features = pd.DataFrame(np.array([getattr(A,attr).flatten() for attr in properties]).T,columns=properties)
        #print(f"Features: \n{features}")

    print("\n")
    print("Iterations for each molecule list: ", normal_iter_list)
    print("Mean iterations to converge: ", np.mean(normal_iter_list))
    print("Deviations from CCSD for each molecule: ", normal_dev)
    print(f"Mean deviation from CCSD: {np.mean(normal_dev)} (Eₕ)")
    print("Times to converge for each molecule: ", normal_time_to_converge_list)
    print("Mean time to converge", np.mean(normal_time_to_converge_list), " seconds")
    print("Next ...\n")


Processing ../machine_learning/data/methane72.xyz
Computing RHF reference.


/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 9 basis functions.
(9, 9)
(8, 8)
Building initial guess...

..initialized CCSD in 0.032 seconds.

CCSD Iteration   0: CCSD correlation = -0.056348025072750   dE =  5.63480E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.056348025072750   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.001 seconds!
CCSD Iteration   0: CCSD correlation = -0.056348025072750   dE =  5.63480E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.071291294011887   dE = -1.49433E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.076122006917623   dE = -4.83071E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.078833106750448   dE = -2.71110E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.079034635041261   dE = -2.01528E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.079052857722892   dE = -1.82227E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation

/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 8 basis functions.
(8, 8)
(7, 7)
Building initial guess...

..initialized CCSD in 0.142 seconds.

CCSD Iteration   0: CCSD correlation = -0.047170836330905   dE =  4.71708E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.047170836330905   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.002 seconds!
CCSD Iteration   0: CCSD correlation = -0.047170836330905   dE =  4.71708E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.059179512124161   dE = -1.20087E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.062849786076141   dE = -3.67027E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.064786074905015   dE = -1.93629E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.064934810124020   dE = -1.48735E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.064945635983222   dE = -1.08259E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation

...

Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 59 basis functions.
(59, 59)
(58, 58)
Building initial guess...

..initialized CCSD in 3.483 seconds.

CCSD Iteration   0: CCSD correlation = -0.168029823329840   dE =  1.68030E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.168029823329840   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.004 seconds!
CCSD Iteration   0: CCSD correlation = -0.168029823329840   dE =  1.68030E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.185481764936227   dE = -1.74519E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.189714754562923   dE = -4.23299E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.191178969725384   dE = -1.46422E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.191402027709772   dE = -2.23058E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.191429578780283   dE = -2.75511E-05   DIIS = 4
CCSD Iteration   6: CCSD correl

/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 50 basis functions.
(50, 50)
(49, 49)
Building initial guess...

..initialized CCSD in 2.063 seconds.

CCSD Iteration   0: CCSD correlation = -0.199402366815540   dE =  1.99402E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.199402366815540   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.004 seconds!
CCSD Iteration   0: CCSD correlation = -0.199402366815540   dE =  1.99402E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.208910662926867   dE = -9.50830E-03   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.212966540144949   dE = -4.05588E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.213951500343623   dE = -9.84960E-04   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.214384650150393   dE = -4.33150E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.214426129789825   dE = -4.14796E-05   DIIS = 4
CCSD Iteration   6: CCSD correl

/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 59 basis functions.
(59, 59)
(58, 58)
Building initial guess...

..initialized CCSD in 3.293 seconds.

CCSD Iteration   0: CCSD correlation = -0.167968407404827   dE =  1.67968E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.167968407404827   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.004 seconds!
CCSD Iteration   0: CCSD correlation = -0.167968407404827   dE =  1.67968E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.185415194220588   dE = -1.74468E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.189640604693657   dE = -4.22541E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.191100796960216   dE = -1.46019E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.191322730637869   dE = -2.21934E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.191350129111014   dE = -2.73985E-05   DIIS = 4
CCSD Iteration   6: CCSD correl

# Random initialisation

## t1 and t2 amplitudes are randomly initiliased

# NOTE: There may not be convergence after 100 iterations, so this fails for some random initialisations for some molecules

In [13]:
basis_sets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

for basis in basis_sets:
    print(f"Basis: {basis}")
    # Random initialisation i.e. all zeros for t1- and t2-amplitudes
    rdm_iter_list = []
    rdm_dev = []
    rdm_time_to_converge_list = []
    
    for fn in filenames[:3]:
        print(f"Processing {fn}")      # import xyz file
        with open(fn,'r') as f:
            text=f.read()
        mol = psi4.geometry(text)                
        
        psi4.core.clean()
        psi4.core.be_quiet()
        
        psi4.set_options({'basis': basis,
                          'scf_type':     'pk',
                          'reference':    'rohf',
                          'mp2_type':     'conv',
                          'e_convergence': 1e-8,
                          'd_convergence': 1e-8})

        rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)        # rhf_e is Hartree-Fock energy 
        scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
        
        A=HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=True)

        # Here is the random way to initialize

        A.t1 = np.random.rand(*A.t1.shape)
        A.t2 = np.random.rand(*A.t2.shape)
        
        rand_0_e = A.compute_energy(iterate=False)      # random initial energy
        rand = A.compute_energy()             # exact CCSD energy
    
        #rand_dev.append(abs(rand_0_e-rand))   # energy deviation from exact CCSD
        # NOTE: Since random may not converge, this above line may return an error
    
        rdmHistory = A.history               # tuple, (iteration_nunber, CCSDcorr_E)
        rdmHistory.insert(0, (0,rand_0_e))
        
        #for i in rdmHistory:
        #    print(i)
        #print("\n")
        
        iterations = len(rdmHistory)     # number of iterations to converge
        #print(f"No. of iterations to converge: {iterations}")
        rdm_iter_list.append(iterations)
        
        rdm_time_to_converge = A.time_to_converge 
        rdm_time_to_converge_list.append(rdm_time_to_converge)
        #print(f"Time to converge: {normal_time_to_converge}")
        
        features = pd.DataFrame(np.array([getattr(A,attr).flatten() for attr in properties]).T,columns=properties)
        #print(f"Features: \n{features}")

    print("\n")
    print("Iterations for each molecule list: ", rdm_iter_list)
    print("Mean iterations to converge: ", np.mean(rdm_iter_list))
    print("Deviations from CCSD for each molecule: ", rdm_dev)
    print(f"Mean deviation from CCSD: {np.mean(rdm_dev)} (Eₕ)")
    print("Times to converge for each molecule: ", rdm_time_to_converge_list)
    print("Mean time to converge", np.mean(rdm_time_to_converge_list), " seconds")
    print("Next ...\n")


Basis: STO-3G
Processing ../machine_learning/data/methane72.xyz
Computing RHF reference.


/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 9 basis functions.
(9, 9)
(8, 8)
Building initial guess...

..initialized CCSD in 0.034 seconds.

CCSD Iteration   0: CCSD correlation = 1.390925348647698   dE = -1.39093E+00   MP2
CCSD Iteration   1: CCSD correlation = 1.390925348647698   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.002 seconds!
CCSD Iteration   0: CCSD correlation = 1.390925348647698   dE = -1.39093E+00   MP2
CCSD Iteration   1: CCSD correlation = 1.104003916407801   dE = -2.86921E-01   DIIS = 0
CCSD Iteration   2: CCSD correlation = 0.880574960225361   dE = -2.23429E-01   DIIS = 1
CCSD Iteration   3: CCSD correlation = 0.706001135843156   dE = -1.74574E-01   DIIS = 2
CCSD Iteration   4: CCSD correlation = 0.449180482097370   dE = -2.56821E-01   DIIS = 3
CCSD Iteration   5: CCSD correlation = 0.062237339456852   dE = -3.86943E-01   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.01

/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 8 basis functions.
(8, 8)
(7, 7)
Building initial guess...

..initialized CCSD in 0.020 seconds.

CCSD Iteration   0: CCSD correlation = 1.143410793016152   dE = -1.14341E+00   MP2
CCSD Iteration   1: CCSD correlation = 1.143410793016152   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.002 seconds!
CCSD Iteration   0: CCSD correlation = 1.143410793016152   dE = -1.14341E+00   MP2
CCSD Iteration   1: CCSD correlation = 1.061790070868393   dE = -8.16207E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = 0.788650340255169   dE = -2.73140E-01   DIIS = 1
CCSD Iteration   3: CCSD correlation = 0.510789546192107   dE = -2.77861E-01   DIIS = 2
CCSD Iteration   4: CCSD correlation = 0.385084791871077   dE = -1.25705E-01   DIIS = 3
CCSD Iteration   5: CCSD correlation = 0.302642145643540   dE = -8.24426E-02   DIIS = 4
CCSD Iteration   6: CCSD correlation = 0.177

...

Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 34 basis functions.
(34, 34)
(33, 33)
Building initial guess...

..initialized CCSD in 0.736 seconds.

CCSD Iteration   0: CCSD correlation = 6.902353606094850   dE = -6.90235E+00   MP2
CCSD Iteration   1: CCSD correlation = 6.902353606094850   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.003 seconds!
CCSD Iteration   0: CCSD correlation = 6.902353606094850   dE = -6.90235E+00   MP2
CCSD Iteration   1: CCSD correlation = 43.366932736626524   dE =  3.64646E+01   DIIS = 0
CCSD Iteration   2: CCSD correlation = 50591.563091364223510   dE =  5.05482E+04   DIIS = 1
CCSD Iteration   3: CCSD correlation = 58296.614919650994125   dE =  7.70505E+03   DIIS = 2
CCSD Iteration   4: CCSD correlation = 12600.591999839489290   dE = -4.56960E+04   DIIS = 3
CCSD Iteration   5: CCSD correlation = 447.296694620166420   dE = -1.21533E+04   DIIS = 4
CCSD Iteration   6: CCSD

/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 29 basis functions.
(29, 29)
(28, 28)
Building initial guess...

..initialized CCSD in 0.471 seconds.

CCSD Iteration   0: CCSD correlation = 4.596291150538116   dE = -4.59629E+00   MP2
CCSD Iteration   1: CCSD correlation = 4.596291150538116   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.002 seconds!
CCSD Iteration   0: CCSD correlation = 4.596291150538116   dE = -4.59629E+00   MP2
CCSD Iteration   1: CCSD correlation = 13.076633207794650   dE =  8.48034E+00   DIIS = 0
CCSD Iteration   2: CCSD correlation = 1769.501887482900656   dE =  1.75643E+03   DIIS = 1
CCSD Iteration   3: CCSD correlation = 1054.954347915507697   dE = -7.14548E+02   DIIS = 2
CCSD Iteration   4: CCSD correlation = 187.030693091049216   dE = -8.67924E+02   DIIS = 3
CCSD Iteration   5: CCSD correlation = 136.336241205519144   dE = -5.06945E+01   DIIS = 4
CCSD Iteration   6: CCSD cor

/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:260: RuntimeWarning: invalid value encountered in divide
  B[:-1, :-1] /= np.abs(B[:-1, :-1]).max()


LinAlgError: Singular matrix

# All Zero initialisation

## t1- and t2- amplitudes are all zeros

In [15]:
basis_sets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

for basis in basis_sets:
    print(f"Basis: {basis}")
    # Random initialisation i.e. all zeros for t1- and t2-amplitudes
    zero_iter_list = []
    zero_dev = []
    zero_time_to_converge_list = []
    
    for fn in filenames[:3]:
        print(f"Processing {fn}")      # import xyz file
        with open(fn,'r') as f:
            text=f.read()
        mol = psi4.geometry(text)                
        
        psi4.core.clean()
        psi4.core.be_quiet()
        
        psi4.set_options({'basis': basis,
                          'scf_type':     'pk',
                          'reference':    'rohf',
                          'mp2_type':     'conv',
                          'e_convergence': 1e-8,
                          'd_convergence': 1e-8})

        rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)        # rhf_e is Hartree-Fock energy 
        scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
        
        A=HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=True)

        # Here is the way to initialise with zeros

        A.t1 = np.zeros(A.t1.shape)
        A.t2 = np.zeros(A.t2.shape)
        
        zero_0_e = A.compute_energy(iterate=False)      # initial energy when t1- and t2-amplitudes are zero
        zero_e = A.compute_energy()             # exact CCSD energy

        zero_dev.append(abs(zero_0_e-zero_e))
        
        zeroHistory = A.history               # tuple, (iteration_nunber, CCSDcorr_E)
        zeroHistory.insert(0, (0,zero_0_e))
        
        #for i in zeroHistory:
        #    print(i)
        #print("\n")
        
        iterations = len(zeroHistory)     # number of iterations to converge
        #print(f"No. of iterations to converge: {iterations}")
        zero_iter_list.append(iterations)
        
        zero_time_to_converge = A.time_to_converge 
        zero_time_to_converge_list.append(zero_time_to_converge)
        #print(f"Time to converge: {normal_time_to_converge}")
        
        features = pd.DataFrame(np.array([getattr(A,attr).flatten() for attr in properties]).T,columns=properties)
        #print(f"Features: \n{features}")

    print("\n")
    print("Iterations for each molecule list: ", zero_iter_list)
    print("Mean iterations to converge: ", np.mean(zero_iter_list))
    print("Deviations from CCSD for each molecule: ", zero_dev)
    print(f"Mean deviation from CCSD: {np.mean(zero_dev)} (Eₕ)")
    print("Times to converge for each molecule: ", zero_time_to_converge_list)
    print("Mean time to converge", np.mean(zero_time_to_converge_list), " seconds")
    print("Next ...\n")


Basis: STO-3G
Processing ../machine_learning/data/methane72.xyz
Computing RHF reference.


/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 9 basis functions.
(9, 9)
(8, 8)
Building initial guess...

..initialized CCSD in 0.041 seconds.

CCSD Iteration   0: CCSD correlation = 0.000000000000000   dE = -0.00000E+00   MP2
CCSD Iteration   1: CCSD correlation = 0.000000000000000   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.004 seconds!
CCSD Iteration   0: CCSD correlation = 0.000000000000000   dE = -0.00000E+00   MP2
CCSD Iteration   1: CCSD correlation = -0.056348025072750   dE = -5.63480E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.071291294011885   dE = -1.49433E-02   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.077955578398195   dE = -6.66428E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.078917088229280   dE = -9.61510E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.079050738114941   dE = -1.33650E-04   DIIS = 4
CCSD Iteration   6: CCSD correlation = 

/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 8 basis functions.
(8, 8)
(7, 7)
Building initial guess...

..initialized CCSD in 0.037 seconds.

CCSD Iteration   0: CCSD correlation = 0.000000000000000   dE = -0.00000E+00   MP2
CCSD Iteration   1: CCSD correlation = 0.000000000000000   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.001 seconds!
CCSD Iteration   0: CCSD correlation = 0.000000000000000   dE = -0.00000E+00   MP2
CCSD Iteration   1: CCSD correlation = -0.047170836330905   dE = -4.71708E-02   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.059179512125440   dE = -1.20087E-02   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.064167266482528   dE = -4.98775E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.064857980064044   dE = -6.90714E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.064950418413835   dE = -9.24383E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = 

...

Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 59 basis functions.
(59, 59)
(58, 58)
Building initial guess...

..initialized CCSD in 3.553 seconds.

CCSD Iteration   0: CCSD correlation = 0.000000000000000   dE = -0.00000E+00   MP2
CCSD Iteration   1: CCSD correlation = 0.000000000000000   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.021 seconds!
CCSD Iteration   0: CCSD correlation = 0.000000000000000   dE = -0.00000E+00   MP2
CCSD Iteration   1: CCSD correlation = -0.168029823329841   dE = -1.68030E-01   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.185481764927904   dE = -1.74519E-02   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.190249072296729   dE = -4.76731E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.191176542008421   dE = -9.27470E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.191411392230300   dE = -2.34850E-04   DIIS = 4
CCSD Iteration   6: CCSD correlati

/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 50 basis functions.
(50, 50)
(49, 49)
Building initial guess...

..initialized CCSD in 2.037 seconds.

CCSD Iteration   0: CCSD correlation = 0.000000000000000   dE = -0.00000E+00   MP2
CCSD Iteration   1: CCSD correlation = 0.000000000000000   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.005 seconds!
CCSD Iteration   0: CCSD correlation = 0.000000000000000   dE = -0.00000E+00   MP2
CCSD Iteration   1: CCSD correlation = -0.199402366815543   dE = -1.99402E-01   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.208910662933748   dE = -9.50830E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.213183311977030   dE = -4.27265E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.214064635365437   dE = -8.81323E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.214387323916410   dE = -3.22689E-04   DIIS = 4
CCSD Iteration   6: CCSD correlati

/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/mnt/c/Users/Maxim/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 59 basis functions.
(59, 59)
(58, 58)
Building initial guess...

..initialized CCSD in 3.879 seconds.

CCSD Iteration   0: CCSD correlation = 0.000000000000000   dE = -0.00000E+00   MP2
CCSD Iteration   1: CCSD correlation = 0.000000000000000   dE =  0.00000E+00   DIIS = 0

CCSD has converged in 0.004 seconds!
CCSD Iteration   0: CCSD correlation = 0.000000000000000   dE = -0.00000E+00   MP2
CCSD Iteration   1: CCSD correlation = -0.167968407404832   dE = -1.67968E-01   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.185415194222800   dE = -1.74468E-02   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.190173907719514   dE = -4.75871E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.191098344094060   dE = -9.24436E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.191332051943370   dE = -2.33708E-04   DIIS = 4
CCSD Iteration   6: CCSD correlati

In [ ]:
#####